# 04 · QC de lote e invariancia

Único QC ejecutable: pre-M-ComBat frente a M-ComBat de referencia. No contiene rutas conjuntas ni correcciones protegidas por etiqueta.

In [ ]:
from pathlib import Path
import os, sys

DATA_PREPARE = Path.cwd().resolve()
while not (DATA_PREPARE / "combat_ref.py").exists():
    if DATA_PREPARE.parent == DATA_PREPARE:
        raise FileNotFoundError("Ejecuta este notebook dentro de data_prepare/.")
    DATA_PREPARE = DATA_PREPARE.parent
os.chdir(DATA_PREPARE)
if str(DATA_PREPARE) not in sys.path:
    sys.path.insert(0, str(DATA_PREPARE))
print("DATA_PREPARE:", DATA_PREPARE)

In [ ]:
import pandas as pd
from qc_fuga import (
    cargar_salidas, comparar_pre_post, tabla_composicion,
    cramer_v_batch_label, reference_pca_scores,
)

d = cargar_salidas(DATA_PREPARE)
qc = comparar_pre_post(**d)
out = DATA_PREPARE / "output_mcombat"

qc.to_csv(out / "qc_batch_pre_post.csv", index=False)
display(qc)

if "platform" in d["meta_all"].columns:
    qc_platform = comparar_pre_post(**d, cohort_col="platform")
    qc_platform.to_csv(out / "qc_platform_pre_post.csv", index=False)
    display(qc_platform)

In [ ]:
meta_all = d["meta_all"]
display(tabla_composicion(meta_all))
print("Cramér V cohorte×clase:", cramer_v_batch_label(meta_all))

invariance = pd.read_csv(out / "verificacion_invariancia.csv")
display(invariance)
assert invariance["max_dif"].max() < 1e-9

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

genes = d["expr_train"].index.intersection(d["expr_test"].index)
train_pre = d["expr_pre"].loc[genes, d["expr_train"].columns]
test_pre = d["expr_pre"].loc[genes, d["expr_test"].columns]
train_post = d["expr_train"].loc[genes]
test_post = d["expr_test"].loc[genes]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for col, (stage, xtr, xte) in enumerate((
    ("pre", train_pre, test_pre), ("post", train_post, test_post),
)):
    ztr, zte, _ = reference_pca_scores(xtr, xte, n_pc=2)
    for row, (split, expr, z) in enumerate((
        ("train", xtr, ztr), ("test", xte, zte),
    )):
        meta = d[f"meta_{split}"].copy()
        if "sample" in meta.columns:
            meta = meta.set_index(meta["sample"].astype(str), drop=False)
        sns.scatterplot(x=z[:,0], y=z[:,1], hue=meta.loc[expr.columns,"dataset"],
                        s=18, linewidth=0, ax=axes[row,col], legend=(row==0 and col==1))
        axes[row,col].set_title(f"{split}: {stage} M-ComBat")
plt.tight_layout()
fig.savefig(out / "qc_pca_batch_pre_post.pdf", bbox_inches="tight")
plt.show()